# Lab 3 — Clean the broken sensor feed
### EP0418 · Topic 4 · 2 practical hours

**You need:** `ep0418.py` and `smart_building_sensors.csv` beside this notebook.

## Putting the assistant's code into a TODO cell

A TODO cell has a line ending in `...`. Replace **only that line** with the
assistant's code, and keep the name on the left (`s`, `clean`, `H`). Delete any
`import` or `read_csv` lines it adds; the data is already loaded. If it used a
different name for the result, add one line at the end to rename it, e.g.
`s = hourly_df`.

---
## How every lab in this module works

You will **not** write this code from scratch. Ask your AI assistant to write it.
Your job is the part it cannot do: deciding what the right answer looks like,
reading what it gave you, and proving it did what you asked.

At the foot of each task is a **verification contract** — a few `check.*` lines.
**You may not edit them.** The AI can write any implementation it likes; it cannot
satisfy these by accident.

If a check fails, read the message. It tells you what was found and what to try
next. Failing checks are the normal state of a lab in progress, not a sign you
are behind.

> ### This dataset is deliberately broken
> **Six** separate faults are hidden in this file. The summary table in the
> next two cells shows three of them, if you read it carefully. The checks catch
> three. **Two you will only find by plotting** — and I am telling you that up
> front so you keep looking.

> **First: File → Save a copy in Drive, and work in your copy.** This page opened from GitHub, and changes here are not saved. Reopen your copy from Google Drive next time.

## Set up Gemini for this notebook (once, 30 seconds)

Open Gemini (the spark icon at the bottom of the notebook). In its chat box, click the **sliders icon** next to **+**, choose **Configure**, paste the text below and save. Leave the mode on **Default**. Colab keeps the instructions with this notebook, so every answer follows the course rules.

```
I am new to coding. Explain any code you write in plain words: what it does to my data.
Before writing code for a bigger task, tell me your plan in plain words.
Never delete or drop rows unless I ask. If a step would remove rows, say so first.
After changing the data, show the row count before and after, and the smallest and largest values.
Codes such as -999 mean a missing reading. Point them out; do not treat them as real numbers.
Do not change any line that starts with check. Those are the lab's locked checks.
If you are not sure, say so. Do not guess.
```

In [ ]:
import os, urllib.request
REPO = "https://raw.githubusercontent.com/StanleySP-eng/EP0418-labs/main/labs/"


def fetch(*names):
    """Download any course file that is not beside the notebook yet."""
    for name in names:
        if os.path.exists(name):
            continue
        try:
            urllib.request.urlretrieve(REPO + name, name)
            print("fetched", name)
        except Exception:
            print(f"Could not download {name}. Upload it with the folder icon on the left.")
            from google.colab import files
            files.upload()


fetch("ep0418.py")
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from ep0418 import check


def filled(*values):
    """True once every TODO slot above has been replaced. Otherwise says so."""
    if any(v is ... or v is None for v in values):
        print("A TODO above is still '...'. Fill it in, then run this cell again.")
        return False
    return True

CSV = "smart_building_sensors.csv"
fetch(CSV)

raw = pd.read_csv(CSV, parse_dates=["timestamp"])
print(f"{len(raw):,} rows, {raw.timestamp.min()} -> {raw.timestamp.max()}")

In [ ]:
print("blank values:", raw.isna().sum().sum())
print("duplicated timestamps:", raw.timestamp.duplicated().sum())
print()
print(raw.describe().T[["min", "mean", "max"]].round(1).to_string())

### Read that table before going further

`isna()` reports zero blanks. But look at the minimum for `humidity_pct`.

**−999 is not a reading.** It is the logger saying "no answer". pandas counts it
as a real number, so it is dragging your mean down right now and `isna()` will
never find it.

How many *other* columns have a min or max that is physically impossible?
Compare each maximum with its mean: a classroom does not use eight times its
usual energy in five minutes.

## Task 1 — Replace the sentinels

> I have a DataFrame `raw` where −999 in `humidity_pct` means "no reading".
> Replace those with NaN. Do not remove any rows.

Read what it gives you. A common wrong answer filters the rows out entirely
instead of marking them blank — which changes your row count, and the contract
below will catch it.

In [ ]:
# TODO(you): `clean` = copy of raw with -999 in humidity_pct replaced by NaN
clean = ...

# ---- verification contract — do not edit ----
check.no_sentinels(clean, cols=["humidity_pct"], sentinel=-999)
check.rows_at_least(clean, 17000)
check.report()

## Task 1b — The same request, with and without the course instructions (10 min)

Your Gemini follows the course instructions in this notebook. Most AI tools you
will meet at work will not. This shows what the instructions were doing for you.

1. Open a **new** notebook (File > New notebook in Drive). It has no custom
   instructions. Open its Gemini panel and ask exactly:
   > Clean the humidity column in my data. The column is humidity_pct and some readings are -999.
2. Ask the Gemini in **this** notebook the same thing, word for word.
3. Compare the two answers:

| Question | Without instructions | With instructions |
|---|---|---|
| Did its code remove any rows? | | |
| Did it show the row count before and after? | | |
| Did it say what −999 means? | | |
| Did it explain its code in plain words? | | |

**Without the instructions, what would you have had to check yourself?**

The instructions are a seatbelt, not a driver. With any AI tool, the checks are
still yours.

## Task 2 — Duplicates and time order

Decide: keep the first, keep the last, or average them? On a logger that retries,
the last is usually the corrected value. Whatever you choose, be ready to justify
it in your log.

In [ ]:
# TODO(you): drop duplicate timestamps and sort into time order
clean = ...

# ---- verification contract — do not edit ----
check.no_duplicate_timestamps(clean, col="timestamp")
check.sorted_in_time(clean, col="timestamp")
check.report()

## Task 3 — Put it on a regular grid

In [ ]:
# TODO(you): resample `clean` onto a regular 5-minute grid called `grid`
grid = ...

# ---- verification contract — do not edit ----
check.regular_cadence(grid, "5min")
check.values_in_range(grid.reset_index(), "temp_c", 15, 40)
check.report()

## Task 4 — Account for every row you lost

Your row count changed three times. Explain why, in your own words.

*"Cleaning removed some rows"* will be rejected. Say **which** rows and **why**.

In [ ]:
# TODO(you): explain, in your own words, every change in the row count
EXPLANATION = "TODO(you): replace with a real explanation"

# ---- verification contract — do not edit ----
check.rows_accounted_for(
    raw, clean, grid,
    names=["raw", "clean", "grid"],
    explain=EXPLANATION,
)
check.report()

## Task 5 — Find the two faults no check can catch

Plot all four sensor channels. Sixty days on one axis hides a fourteen-hour
flat line, so also plot a few days at a time, or ask your assistant for the
longest run of identical consecutive readings in each channel and the date it
starts.

One fault is a sensor that **stops changing** — it produces no outliers at all,
just a flat line, which looks perfectly well-behaved to any alarm.

The other is a **slow drift** that makes a fixed threshold fire more and more
often even though the room never changed.

In [ ]:
# TODO(you): plot temp_c, humidity_pct, co2_ppm and energy_kwh over time
...

---
## Read before you run

An assistant was asked to resample the cleaned sensor data onto a 5-minute grid so that `no_missing` passes. It wrote:

**Do not run it yet.** Read it, answer the two questions in the cell below,
and only then remove the `#` and run it to see whether you were right.

What does the second line do to the 5½-hour outage on 11 June? Will any of the contracts in this lab fail?

Write down how many readings the second line invents.

In [ ]:
# --- AI-generated. Read it first. ---
# grid = clean.set_index("timestamp").resample("5min").mean()
# grid = grid.interpolate()

**What is wrong with it:**

**What it would produce, and what it should produce:**

---
## Before you leave

`verification_log.md` — one entry per AI mistake you caught today, and one
**Decisions** line for each fault you handled: what you did about the duplicates,
the −999s and the outage gap, and why.

Then answer: **which date is the stuck sensor on, and which channel?** Bring the
answer to the next session.